# Taxa de transmissão e decaimento (mu)

Regressão por avaliação semanal, pelo método de Chang & de Jong (2023): a probabilidade de uma
vaca saudável virar caso na semana depende da exposição somada nos 7 dias antes da avaliação.

- **sem fundo**: `p = 1 - exp(-beta * exp)`, o GLM do artigo. Supõe que toda infecção vem da
  exposição no piso, então semanas com exposição zero ficam de fora.
- **com fundo**: `p = 1 - exp(-(lambda0 * 7 + beta * exp))`. `lambda0` é o risco por dia que não
  vem da exposição (ordenha, vacas fora do estudo, casos não detectados).

Duas fontes de exposição:

- **PeDViS-BDD** (arquivo do Gustavo): por vaca, a partir do rastreamento. Só existe para mu = 0.07.
- **por baia** (`src/barn.py`): mesma contaminação para todas as vacas da baia, calculada com as
  equações do artigo para qualquer mu. É outro modelo, não reproduz o PeDViS-BDD.

Lê os resultados gravados por `python -m src.glm`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.barn import add_barn_exposure
from src.daily import build_daily
from src.glm import at_risk_weeks, fit_with_background
from src.load import PROCESSED_DIR, load_exposure
from src.weekly import build_weekly

pedvis = pd.read_parquet(PROCESSED_DIR / "glm_pedvis_mu0.07.parquet")
profile = pd.read_parquet(PROCESSED_DIR / "glm_barn_profile.parquet")
profile["dAIC"] = profile["aic"] - profile.groupby("model")["aic"].transform("min")

In [ ]:
# cores e estilo dos gráficos
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "font.family": ["Segoe UI", "DejaVu Sans"], "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlecolor": INK,
})
COLUMNS = ["model", "weeks", "cases", "beta", "beta_low", "beta_high", "lambda0", "aic", "p_exposure"]

## PeDViS-BDD, mu = 0.07

Com as semanas que têm os 7 dias de exposição e com os dias disponíveis (análise de
sensibilidade, porque a falta de exposição não é aleatória). `p_exposure` testa se a exposição
explica alguma coisa além do risco de fundo; 0,5 quer dizer que o melhor beta é zero.

In [ ]:
pedvis.set_index("weeks_used")[COLUMNS]

## Modelo por baia: perfil de mu

Para cada mu, o mesmo modelo é ajustado nas mesmas semanas, e a diferença de AIC para o melhor
mu (dAIC) mostra quais valores os dados aceitam: abaixo de 2, o mu é compatível com os dados.
O gráfico corta o dAIC em 20 para mostrar a região que importa.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, sharey=True)
for ax, (model, color) in zip(axes, [("sem fundo", BLUE), ("com fundo", ORANGE)]):
    g = profile[profile["model"].eq(model)].sort_values("mu")
    ax.plot(g["mu"], g["dAIC"].clip(upper=20), color=color, lw=1.5, marker="o", ms=4)
    ax.axhline(2, color=MUTED, lw=0.8, ls="--")
    ax.axvline(0.07, color=MUTED, lw=0.8)
    best = g.loc[g["aic"].idxmin()]
    if g["dAIC"].lt(2).all():
        ax.set_title(f"{model}: nenhum mu se destaca (dAIC < 2 em todos)")
    else:
        ax.set_title(f"{model}: melhor mu = {best['mu']:.3g} por dia")
    if g["dAIC"].gt(20).any():
        ax.annotate("acima de 20: cortado", (g["mu"].max(), 20), xytext=(0, -6),
                    textcoords="offset points", ha="right", va="top", color=INK_2)
    ax.annotate("dAIC = 2", (g["mu"].max(), 2), xytext=(0, 3), textcoords="offset points",
                ha="right", va="bottom", color=INK_2)
    ax.set_ylabel("dAIC")
axes[0].annotate("mu do arquivo do Gustavo (0.07)", (0.07, 20), xytext=(4, -2),
                 textcoords="offset points", va="top", color=INK_2)
axes[1].set_xscale("log")
axes[1].set_xlabel("mu (decaimento por dia)")
axes[1].set_ylim(0, 21)
fig.tight_layout()

In [ ]:
def accepted(g):
    ok = g[g["dAIC"] < 2]
    best = g.loc[g["aic"].idxmin()]
    return pd.Series({
        "melhor mu": best["mu"], "mu aceito de": ok["mu"].min(), "até": ok["mu"].max(),
        "meia-vida no melhor mu (dias)": np.log(2) / best["mu"],
        **best[COLUMNS[1:]],
    })

profile.groupby("model").apply(accepted, include_groups=False)

## Comparação nas mesmas semanas

Para comparar o PeDViS-BDD com o modelo por baia, os dois precisam ser ajustados nas mesmas
semanas: as que têm os 7 dias de exposição do PeDViS-BDD e alguma exposição por baia. O modelo
por baia só existe nos dias do arquivo de infecção, então algumas semanas do PeDViS-BDD ficam de
fora. Usa o modelo com fundo, que aceita todas as semanas. AIC menor é melhor; diferenças abaixo
de 2 não separam os modelos.

In [ ]:
daily = build_daily(MU := "0.07")
pedvis_weeks = at_risk_weeks(build_weekly(daily, load_exposure(MU)))
best_barn = profile.loc[profile.query("model == 'com fundo'")["aic"].idxmin(), "mu"]
barn_weeks = {
    mu: at_risk_weeks(build_weekly(add_barn_exposure(daily, mu)), complete=False)
    for mu in sorted({0.07, best_barn})
}

keys = pedvis_weeks[["Agent", "date"]]
for weeks in barn_weeks.values():
    keys = keys.merge(weeks[["Agent", "date"]])
print(f"semanas comparadas: {len(keys)} de {len(pedvis_weeks)} do PeDViS-BDD")

rows = [fit_with_background(keys.merge(pedvis_weeks)) | {"exposure": "PeDViS-BDD, mu = 0.07"}]
for mu, weeks in barn_weeks.items():
    rows.append(fit_with_background(keys.merge(weeks)) | {"exposure": f"por baia, mu = {mu:.3g}"})

comparison = pd.DataFrame(rows).set_index("exposure")[COLUMNS[1:]]
comparison["dAIC"] = comparison["aic"] - comparison["aic"].min()
comparison

## Como ler

- **Sem fundo**, o modelo por baia aponta um mu na faixa do gráfico de cima. Esse resultado só
  vale se toda infecção vier da contaminação do piso da baia.
- **Com fundo**, se o perfil é plano e `p_exposure` é alto, os dados não distinguem a transmissão
  pelo piso de um risco constante: o número de casos é pequeno para estimar beta e mu juntos.
- A comparação nas mesmas semanas diz se a exposição de cada vaca (PeDViS-BDD) explica os casos
  melhor do que a exposição média da baia.